# Minería de texto con LDA: descubrimiento de temas

## Contexto práctico

Una empresa recibe comentarios de clientes por correo, chat y encuestas. Los comentarios no tienen categorías y el equipo quiere descubrir cuáles son los temas más frecuentes para decidir dónde invertir esfuerzos de mejora.

En este notebook usaremos **LDA**, una técnica de aprendizaje no supervisado que descubre temas latentes a partir de las palabras que aparecen juntas en los documentos.

El resultado esperado no es una etiqueta perfecta escrita previamente. Es un mapa exploratorio de temas que ayude a formular preguntas y priorizar análisis.

### La idea en una frase

Imagine que recibe una caja con muchos comentarios mezclados. LDA intenta separar esa caja en grupos temáticos observando qué palabras suelen aparecer juntas. Después, una persona revisa esos grupos y les pone nombres comprensibles.

Por ejemplo, si en varios comentarios aparecen juntos paquete, envío y entrega, podemos interpretar que existe un tema relacionado con logística.

## Objetivos de aprendizaje

- Entender qué problema resuelve LDA.
- Preparar documentos para un modelo de tópicos.
- Representar textos como una matriz de conteos.
- Entrenar LDA y seleccionar un número razonable de tópicos.
- Interpretar las palabras principales de cada tópico.
- Asignar un tópico dominante a cada comentario.
- Analizar la mezcla de temas en un documento.
- Traducir los resultados a conclusiones de negocio.

El dataset está incluido dentro del notebook como una lista de comentarios. No es necesario cargar archivos externos.

### ¿Cuál es el beneficio para la empresa?

Antes de automatizar una clasificación, la empresa necesita saber qué asuntos aparecen realmente. LDA ayuda a descubrirlos cuando todavía no existe un catálogo de categorías.

El resultado puede apoyar decisiones como: crear nuevas categorías de atención, priorizar problemas frecuentes, diseñar preguntas de encuesta, asignar personal o investigar un aumento inesperado de quejas.

## Diferencia frente a otras técnicas

- Clasificación supervisada: aprende categorías que ya conocemos.
- NER: extrae entidades como personas, ciudades o fechas.
- Embeddings: busca textos con significado similar.
- LDA: descubre grupos de palabras que aparecen juntas y los interpreta como temas.

LDA es especialmente útil cuando todavía no sabemos qué categorías existen.

## 1. Importar bibliotecas

Usaremos pandas para organizar los comentarios, scikit-learn para vectorizar y entrenar LDA, y matplotlib/seaborn para visualizar resultados. No se necesitan descargas adicionales en Google Colab.

In [ ]:
import re
import unicodedata
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
RANDOM_STATE = 42
pd.set_option('display.max_colwidth', 140)

## 2. Dataset incluido en el notebook

Cada comentario representa la opinión de un cliente. No incluimos una columna de tema verdadero porque precisamente queremos descubrir los temas.

La variable canal solamente aporta contexto; no se utilizará para entrenar LDA. Más adelante podremos observar si ciertos temas son más frecuentes en un canal.

In [ ]:
datos = [
    {'id': 'C001', 'canal': 'chat', 'comentario': 'La aplicación se cierra cuando intento iniciar sesión'},
    {'id': 'C002', 'canal': 'correo', 'comentario': 'No puedo entrar a mi cuenta porque la contraseña no funciona'},
    {'id': 'C003', 'canal': 'encuesta', 'comentario': 'El código de acceso nunca llega a mi correo'},
    {'id': 'C004', 'canal': 'chat', 'comentario': 'Mi usuario está bloqueado y no puedo ingresar'},
    {'id': 'C005', 'canal': 'correo', 'comentario': 'El portal muestra error al guardar mis datos'},
    {'id': 'C006', 'canal': 'encuesta', 'comentario': 'La página tarda demasiado y se queda cargando'},
    {'id': 'C007', 'canal': 'chat', 'comentario': 'La aplicación está lenta desde esta mañana'},
    {'id': 'C008', 'canal': 'correo', 'comentario': 'El sistema no responde cuando intento actualizar la cuenta'},
    {'id': 'C009', 'canal': 'encuesta', 'comentario': 'El botón de pago no responde en la página'},
    {'id': 'C010', 'canal': 'chat', 'comentario': 'El portal tiene una falla y no permite continuar'},
    {'id': 'C011', 'canal': 'correo', 'comentario': 'No reconozco un cargo en mi factura'},
    {'id': 'C012', 'canal': 'encuesta', 'comentario': 'Me cobraron dos veces el mismo servicio'},
    {'id': 'C013', 'canal': 'chat', 'comentario': 'El importe de la factura no coincide con el contrato'},
    {'id': 'C014', 'canal': 'correo', 'comentario': 'Necesito una factura con mis datos fiscales'},
    {'id': 'C015', 'canal': 'encuesta', 'comentario': 'Quiero aclarar un cobro que aparece en mi cuenta'},
    {'id': 'C016', 'canal': 'chat', 'comentario': 'El pago fue rechazado aunque tengo saldo'},
    {'id': 'C017', 'canal': 'correo', 'comentario': 'Mi paquete todavía no llega a mi domicilio'},
    {'id': 'C018', 'canal': 'encuesta', 'comentario': 'Quiero rastrear el envío porque está retrasado'},
    {'id': 'C019', 'canal': 'chat', 'comentario': 'El pedido llegó incompleto y falta un producto'},
    {'id': 'C020', 'canal': 'correo', 'comentario': 'Recibí el paquete dañado durante la entrega'},
    {'id': 'C021', 'canal': 'encuesta', 'comentario': 'Necesito conocer la fecha de entrega de mi pedido'},
    {'id': 'C022', 'canal': 'chat', 'comentario': 'El repartidor no encontró mi domicilio'},
    {'id': 'C023', 'canal': 'correo', 'comentario': 'Quiero cancelar mi suscripción mensual'},
    {'id': 'C024', 'canal': 'encuesta', 'comentario': 'Ya no deseo continuar con el plan contratado'},
    {'id': 'C025', 'canal': 'chat', 'comentario': 'Solicito dar de baja el servicio antes del próximo cobro'},
    {'id': 'C026', 'canal': 'correo', 'comentario': 'Quiero eliminar definitivamente mi cuenta'},
    {'id': 'C027', 'canal': 'encuesta', 'comentario': 'Necesito terminar el contrato del servicio'},
    {'id': 'C028', 'canal': 'chat', 'comentario': 'Solicito confirmar la cancelación de mi plan'},
    {'id': 'C029', 'canal': 'correo', 'comentario': 'La devolución todavía no aparece en mi cuenta'},
    {'id': 'C030', 'canal': 'encuesta', 'comentario': 'Quiero saber cuándo recibiré el reembolso del cobro'},
    {'id': 'C031', 'canal': 'chat', 'comentario': 'El reembolso de mi compra sigue pendiente'},
    {'id': 'C032', 'canal': 'correo', 'comentario': 'Solicito revisar una devolución que no ha llegado'}
]

df = pd.DataFrame(datos)
df.to_csv('comentarios_clientes_lda.csv', index=False, encoding='utf-8')
print(f'Comentarios cargados: {len(df)}')
display(df.head())

### Interpretación inicial

El dataset contiene varios temas diseñados para que el resultado sea fácil de interpretar: acceso, fallas técnicas, facturación, entregas, cancelaciones y reembolsos. En datos reales, estos temas no estarían definidos de antemano y podrían mezclarse o aparecer nuevos.

In [ ]:
display(df['canal'].value_counts().to_frame('comentarios'))
sns.countplot(data=df, x='canal', order=df['canal'].value_counts().index)
plt.title('Comentarios por canal')
plt.xlabel('Canal')
plt.ylabel('Cantidad')
plt.show()

## 3. Limpiar el texto

La limpieza reduce variaciones que no aportan significado: minúsculas, acentos, signos y espacios repetidos. No eliminaremos palabras de forma indiscriminada; primero conservaremos el contenido y después usaremos una lista explícita de palabras vacías en español.

En un proyecto real, conviene revisar manualmente si términos como no, nunca o todavía son relevantes para el objetivo.

In [ ]:
def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

df['texto_limpio'] = df['comentario'].apply(limpiar_texto)
display(df[['comentario', 'texto_limpio']].head(8))

## 4. Crear la matriz de documentos y palabras

LDA trabaja con conteos de palabras. CountVectorizer crea una matriz donde cada fila es un comentario y cada columna es una palabra. El valor indica cuántas veces aparece esa palabra.

min_df=1 conserva palabras que aparecen al menos una vez. max_df=0.90 elimina palabras presentes en casi todos los documentos. Los bigramas no son indispensables en este ejemplo; usamos palabras individuales para que la interpretación sea sencilla.

In [ ]:
palabras_vacias = [
    'el', 'la', 'los', 'las', 'un', 'una', 'unos', 'unas', 'de', 'del',
    'en', 'a', 'por', 'para', 'con', 'mi', 'mis', 'que', 'y', 'no',
    'se', 'su', 'sus', 'lo', 'me', 'es', 'al', 'como', 'cuando'
]

vectorizador = CountVectorizer(
    stop_words=palabras_vacias,
    min_df=1,
    max_df=0.90,
    token_pattern=r'(?u)\b[a-zA-ZáéíóúüñÁÉÍÓÚÜÑ][a-zA-ZáéíóúüñÁÉÍÓÚÜÑ]+\b'
)
matriz_documentos = vectorizador.fit_transform(df['texto_limpio'])
terminos = vectorizador.get_feature_names_out()
print(f'Matriz: {matriz_documentos.shape[0]} documentos x {matriz_documentos.shape[1]} palabras')
display(pd.DataFrame(matriz_documentos.toarray(), columns=terminos).head())

### Interpretación de la matriz

La matriz es una traducción numérica del lenguaje. LDA no comprende el texto como una persona; observa patrones de coaparición. Si palabras como paquete, envío y entrega aparecen repetidamente en los mismos documentos, es probable que formen parte de un tópico.

## 5. Entrenar el modelo LDA

LDA supone que cada documento mezcla varios tópicos y que cada tópico es una distribución de palabras. Por ejemplo, un comentario puede ser principalmente de facturación, pero contener también una referencia a un pago.

n_components define cuántos tópicos queremos descubrir. Usaremos seis porque el ejercicio contiene aproximadamente seis asuntos. En un caso real probaríamos varios valores y combinaríamos métricas con interpretación humana.

In [ ]:
NUM_TOPICOS = 6
lda = LatentDirichletAllocation(
    n_components=NUM_TOPICOS,
    random_state=RANDOM_STATE,
    learning_method='batch',
    max_iter=50,
    doc_topic_prior=0.1,
    topic_word_prior=0.1
)
distribucion_documentos = lda.fit_transform(matriz_documentos)
print(f'Modelo entrenado con {NUM_TOPICOS} tópicos')
print(f'Perplejidad del modelo: {lda.perplexity(matriz_documentos):.2f}')

### ¿Qué acaba de hacer el modelo?

El modelo recibió una tabla de números, no una tabla con nombres como entregas o facturación. A partir de las palabras que aparecen juntas, creó seis grupos matemáticos. Todavía no sabemos qué significa cada grupo: esa interpretación se hará en el siguiente bloque.

Entrada: comentarios convertidos en conteos. Resultado: seis distribuciones de palabras y una distribución de tópicos para cada comentario.

### ¿Qué significa la perplejidad?

La perplejidad es una medida interna de qué tan bien el modelo explica los documentos; valores menores suelen ser preferibles al comparar modelos sobre datos comparables. No debe utilizarse sola: un modelo puede reducir la perplejidad y producir tópicos difíciles de interpretar.

La calidad de un tópico se juzga combinando coherencia de sus palabras, utilidad para el negocio y revisión humana.

### Ejemplo sencillo de lectura

Supongamos que un tópico muestra estas palabras: paquete, envío, entrega, pedido y domicilio. No significa que cada comentario contenga todas esas palabras. Significa que, en conjunto, esas palabras son las señales más asociadas con ese tópico.

El analista podría nombrarlo Tema de entregas. Ese nombre no lo inventó automáticamente el algoritmo; lo asignó una persona después de revisar la evidencia.

## 6. Interpretar las palabras principales de cada tópico

LDA aprende pesos para cada palabra dentro de cada tópico. La función siguiente muestra las palabras con mayor peso y nos permite asignar una descripción humana al tópico.

In [ ]:
def mostrar_topicos(modelo, terminos, n_palabras=8):
    filas = []
    for numero, pesos in enumerate(modelo.components_):
        indices = pesos.argsort()[-n_palabras:][::-1]
        palabras = [terminos[i] for i in indices]
        filas.append({'topico': numero, 'palabras_principales': ', '.join(palabras)})
    return pd.DataFrame(filas)

topicos_df = mostrar_topicos(lda, terminos)
display(topicos_df)

## 6.1 Poner nombres comprensibles a los tópicos

LDA devuelve números, pero un número no es un nombre de negocio. Primero revisamos las palabras principales y después asignamos una descripción humana.

La celda siguiente usa palabras guía para proponer un nombre. Es una ayuda didáctica; en un proyecto real el analista debe confirmar el nombre leyendo también varios comentarios representativos. Si no hay suficientes palabras coincidentes, el resultado queda como tópico por revisar.

In [ ]:
palabras_guia = {
    'Acceso y cuenta': {'acceso', 'cuenta', 'contrasena', 'codigo', 'ingresar', 'usuario'},
    'Fallas técnicas': {'aplicacion', 'portal', 'pagina', 'sistema', 'error', 'cargando', 'lenta'},
    'Facturación y pagos': {'factura', 'cargo', 'cobro', 'pago', 'importe', 'fiscales'},
    'Entregas y pedidos': {'paquete', 'envio', 'pedido', 'entrega', 'domicilio', 'repartidor'},
    'Cancelaciones': {'cancelar', 'cancelacion', 'suscripcion', 'baja', 'plan', 'contrato'},
    'Reembolsos': {'devolucion', 'reembolso', 'compra', 'pendiente'}
}

def proponer_nombre_topico(pesos, terminos, numero):
    indices = pesos.argsort()[-10:][::-1]
    palabras_topico = set(terminos[indices])
    puntajes = {nombre: len(palabras_topico & guia) for nombre, guia in palabras_guia.items()}
    nombre, puntaje = max(puntajes.items(), key=lambda elemento: elemento[1])
    if puntaje == 0:
        return f'Tópico {numero} - revisar'
    return nombre

nombres_por_indice = {
    numero: proponer_nombre_topico(lda.components_[numero], terminos, numero)
    for numero in range(NUM_TOPICOS)
}

topicos_df['nombre_propuesto'] = topicos_df['topico'].map(nombres_por_indice)
display(topicos_df)

### Diccionario de temas para lectura humana

La tabla siguiente traduce el número técnico a un nombre claro y una explicación breve. El número sirve para el modelo; el nombre sirve para comunicar el resultado al equipo de negocio.

In [ ]:
descripciones = {
    'Acceso y cuenta': 'Problemas para entrar, usar la contraseña o desbloquear el usuario.',
    'Fallas técnicas': 'Errores, lentitud, botones que no responden o pantallas que no cargan.',
    'Facturación y pagos': 'Facturas, cargos, cobros duplicados, importes o pagos rechazados.',
    'Entregas y pedidos': 'Paquetes retrasados, envíos, domicilios, entregas incompletas o dañadas.',
    'Cancelaciones': 'Bajas de suscripciones, planes, cuentas o contratos.',
    'Reembolsos': 'Devoluciones y reembolsos que están pendientes o no aparecen.'
}

catalogo_topicos = topicos_df[['topico', 'nombre_propuesto', 'palabras_principales']].copy()
catalogo_topicos['descripcion'] = catalogo_topicos['nombre_propuesto'].map(descripciones).fillna(
    'Tema no identificado con claridad; revisar sus palabras y comentarios.'
)
catalogo_topicos = catalogo_topicos.rename(columns={
    'topico': 'numero_tecnico',
    'nombre_propuesto': 'nombre_claro'
})
display(catalogo_topicos[['numero_tecnico', 'nombre_claro', 'descripcion', 'palabras_principales']])

### Cómo interpretar los tópicos

Un tópico no tiene nombre automático. El analista debe leer sus palabras principales y asignar una etiqueta comprensible, por ejemplo “entregas” o “facturación”.

La interpretación debe considerar el conjunto de palabras, no una sola. Si aparecen aplicación, error, página y sistema, una descripción razonable podría ser “fallas técnicas”. Si un tópico mezcla palabras sin relación, puede ser necesario ajustar el número de tópicos, la limpieza o el dataset.

In [ ]:
for numero, pesos in enumerate(lda.components_):
    indices = pesos.argsort()[-8:][::-1]
    palabras = [terminos[i] for i in indices]
    valores = pesos[indices]
    plt.figure(figsize=(7, 3))
    sns.barplot(x=valores, y=palabras, color='#4472C4')
    plt.title(f'Tópico {numero}: palabras principales')
    plt.xlabel('Peso aprendido por LDA')
    plt.ylabel('Palabra')
    plt.show()

## 7. Asignar el tópico dominante a cada comentario

distribucion_documentos contiene una fila por comentario y una columna por tópico. Cada fila suma aproximadamente 1 y representa la mezcla de temas del documento.

El tópico dominante es el que tiene la mayor probabilidad dentro de un comentario. Esta simplificación ayuda a resumir, pero no debemos olvidar que un documento puede mezclar varios temas.

In [ ]:
for numero in range(NUM_TOPICOS):
    df[f'topico_{numero}'] = distribucion_documentos[:, numero]

df['topico_dominante'] = distribucion_documentos.argmax(axis=1)
df['nombre_topico_dominante'] = df['topico_dominante'].map(nombres_por_indice)
df['descripcion_topico_dominante'] = df['nombre_topico_dominante'].map(descripciones).fillna(
    'Tema no identificado con claridad; revisar manualmente.'
)
df['peso_topico_dominante'] = distribucion_documentos.max(axis=1).round(3)
display(df[['id', 'comentario', 'topico_dominante', 'nombre_topico_dominante', 'descripcion_topico_dominante', 'peso_topico_dominante']].head(12))

### Interpretación de la asignación

El tópico dominante es una etiqueta exploratoria. Un peso alto indica que el modelo ve un tema principal claro; un peso bajo indica que el comentario mezcla temas o que el modelo no tiene suficiente evidencia.

Para mejorar la utilidad, una persona debe revisar varios comentarios representativos de cada tópico y asignar nombres de negocio después de observarlos.

In [ ]:
resumen_topicos = df.groupby('topico_dominante').agg(
    comentarios=('id', 'count'),
    peso_promedio=('peso_topico_dominante', 'mean')
).reset_index()
resumen_topicos['nombre_claro'] = resumen_topicos['topico_dominante'].map(nombres_por_indice)
display(resumen_topicos[['topico_dominante', 'nombre_claro', 'comentarios', 'peso_promedio']])

sns.barplot(data=resumen_topicos, x='topico_dominante', y='comentarios', color='#70AD47')
plt.title('Comentarios asignados a cada tópico dominante')
plt.xlabel('Tópico descubierto')
plt.ylabel('Cantidad de comentarios')
plt.show()

## 8. Revisar documentos representativos

Una práctica importante es leer los comentarios con mayor peso para cada tópico. Son los casos que mejor muestran el significado aprendido por el modelo.

In [ ]:
representativos = []
for numero in range(NUM_TOPICOS):
    candidatos = df[df['topico_dominante'] == numero].sort_values(
        f'topico_{numero}', ascending=False
    ).head(3)
    representativos.append(candidatos[['id', 'comentario', 'peso_topico_dominante']])

representativos_df = pd.concat(representativos, ignore_index=True)
display(representativos_df)

### Interpretación de casos representativos

Si los tres comentarios principales de un tópico tratan el mismo asunto, el tópico es interpretable. Si mezclan problemas distintos, puede ocurrir que haya demasiados pocos documentos, que las palabras sean ambiguas o que el número de tópicos no sea adecuado.

Esta revisión humana es indispensable: LDA entrega patrones estadísticos, pero el significado final lo asigna el analista con conocimiento del negocio.

### ¿Por qué esta revisión genera valor?

La lista de palabras sola no es una decisión. El valor aparece cuando la empresa convierte un tópico en una acción. Por ejemplo:

- muchos comentarios sobre entregas pueden justificar revisar al proveedor logístico;
- muchos comentarios sobre fallas técnicas pueden justificar priorizar una corrección;
- muchos comentarios sobre reembolsos pueden revelar un problema en el proceso de pagos.

LDA ayuda a descubrir dónde mirar. Después se necesitan datos de volumen, fechas, costos y satisfacción para decidir qué atender primero.

## 9. Probar una nueva consulta

LDA también puede transformar un comentario nuevo usando el mismo vectorizador y el mismo modelo. Así podemos explorar a qué tema se parece una consulta, sin entrenar nuevamente.

In [ ]:
nueva_consulta = 'El envío está retrasado y todavía no recibo mi paquete'
consulta_limpia = limpiar_texto(nueva_consulta)
consulta_matriz = vectorizador.transform([consulta_limpia])
consulta_distribucion = lda.transform(consulta_matriz)[0]
topico_consulta = consulta_distribucion.argmax()

print(f'Consulta: {nueva_consulta}')
print(f'Tópico dominante: {topico_consulta}')
print(f'Nombre propuesto: {nombres_por_indice[topico_consulta]}')
print(f'Peso del tópico dominante: {consulta_distribucion[topico_consulta]:.3f}')
display(pd.DataFrame({'topico': range(NUM_TOPICOS), 'peso': consulta_distribucion.round(3)}))

### Interpretación de la consulta

La distribución permite ver si la consulta pertenece claramente a un tema o si comparte señales con varios. Una consulta de entrega debería concentrar mayor peso en el tópico cuyas palabras incluyen envío, paquete, pedido o entrega.

Si todos los pesos son parecidos, el sistema debería evitar una conclusión fuerte y solicitar revisión o más contexto.

## 10. Del resultado técnico a una decisión

El flujo práctico sería:

1. Reunir comentarios sin clasificar.
2. Ejecutar LDA para descubrir temas.
3. Leer las palabras y los comentarios representativos.
4. Poner nombres de negocio a los tópicos.
5. Contar la frecuencia de cada tema por periodo, canal o producto.
6. Elegir una acción: investigar, corregir, priorizar o crear una nueva categoría.

LDA es principalmente una herramienta de descubrimiento. No debe interpretarse como una verdad definitiva ni como una sustitución del conocimiento del negocio.

## 11. Conclusiones generales

1. LDA descubre temas a partir de la coaparición de palabras, sin necesitar etiquetas previas.
2. Cada documento puede contener una mezcla de temas; el tópico dominante es solo un resumen.
3. Las palabras principales permiten describir los tópicos, pero necesitan interpretación humana.
4. La perplejidad ayuda a comparar modelos, aunque no reemplaza la coherencia ni la utilidad empresarial.
5. Los documentos representativos son esenciales para validar si cada tópico tiene sentido.
6. Los temas descubiertos pueden apoyar priorización, análisis de causas y diseño de indicadores.
7. En producción se recomienda monitorear nuevos vocabularios, revisar comentarios ambiguos y actualizar el modelo periódicamente.

### Conclusión ejecutiva

LDA es una herramienta exploratoria para convertir una colección de comentarios sin clasificar en un mapa de temas. Su mayor valor aparece al inicio de una investigación: ayuda a descubrir qué está ocurriendo, cuantificar la demanda por asunto y decidir qué categorías formales conviene crear después.